# basketStatsNB1 — last season's player stats, per game

Pipeline: **team -> last season's games -> each game's box score -> our team's player rows**, stacked into one DataFrame.

**Important finding while building this:** the team page's season selector (e.g. `?season=2526`) does *not* actually change what's returned -- verified three different ways (raw HTML, the SvelteKit `__data.json` API, and the rendered page) that its `liveGames` field only ever contains the *current* season's fixtures, regardless of the `season` query param. That's a real limitation of that route, not a scraping bug.

The reliable way to get last season's games is via the **competition page** instead, whose GUID encodes the season directly (e.g. `BVBL26279130OVHSE31A` for 2026-27 vs `BVBL25269130OVHSE31B` for 2025-26) and returns its match schedule as plain embedded JSON -- no browser needed, just `requests`. One wrinkle: a team's pool letter can change between seasons (HAANTJES HSE2 was in pool B in 2025-26 but is in pool A this season), so `get_last_season_games` scans candidate pool letters rather than assuming the same one.

Functions 2 and 3 still use Playwright (headless Chromium), since the game overview and box-score pages (`/games/{guid}`, `/games/{guid}/home`, `/games/{guid}/away`) render their `Tabulator` tables client-side after load and aren't available as plain embedded JSON. Verified against a real finished game (`BVBL25269130BOVHSEVR03`): both box-score pages return real per-player rows with fields `Number, Name, NormalizedMinutes, PlusMinus, Faults, TotalScore, ThreePointers, FieldGoals, FreeThrows, TotalScorePerMinute, PlusMinusPerMinute` (there's also a `Guid` column, but it renders as a plain icon with no data, so it's dropped).

In [20]:
import re
import json
import requests
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from playwright.sync_api import sync_playwright

BASE_URL = "https://app.basketballstatsvlaanderen.be"

In [2]:
CLUB_GUID = "BVBL1037"
TEAM_GUID = "BVBL1037HSE  2"
HEADLESS = True

## 1. `get_last_season_games` — a team's most recently completed season (via the competition page, no browser needed)

In [22]:
def _previous_season(season: str) -> str:
    """'2627' -> '2526'."""
    start, end = int(season[:2]), int(season[2:])
    return f"{start - 1:02d}{end - 1:02d}"


def _get_competition_matches(competition_guid: str) -> pd.DataFrame:
    """A competition page embeds its match schedule as plain JSON in a <script> tag
    (`{"body": "<json>"}`), so a plain GET is enough -- no browser needed."""
    r = requests.get(f"{BASE_URL}/competitions/{competition_guid}", timeout=30)
    r.raise_for_status()
    for script in re.findall(r"<script[^>]*>(.*?)</script>", r.text, re.DOTALL):
        try:
            payload = json.loads(script)
            body = json.loads(payload["body"])
        except (json.JSONDecodeError, KeyError, TypeError):
            continue
        if isinstance(body, list) and body and "HomeTeam" in body[0]:
            return pd.DataFrame(body)
    return pd.DataFrame()


def get_last_season_games(club_guid: str, team_guid: str,
                           tiers=("11", "21", "31", "41", "51", "61"),
                           pool_letters: str = "ABCDEFGH") -> tuple[str, str, pd.DataFrame]:
    """Return (team_name, last_season, games_df) for a team's most recently completed season.

    The team page's own season selector doesn't work (verified: its `liveGames` data never
    changes regardless of the `season` query param) -- so this instead derives last season's
    competition GUID from the team's current one (the season is encoded right in the GUID,
    e.g. 'BVBL26279130OVHSE31A' -> 'BVBL25269130OVHSE<tier><pool>') and scans candidate
    tier + pool-letter combinations, since a team's division can change between seasons in
    either dimension -- not just a different pool letter (e.g. '31A' -> '31B'), but a
    different tier entirely after a promotion/relegation (e.g. '41A' -> '31B').
    """
    info = requests.get(f"{BASE_URL}/api/clubs/{club_guid}/{team_guid}", timeout=20).json()
    team_name = info["team"]["Name"]
    competitions = info["team"]["Competitions"]

    league_comp = next((c for c in competitions if "beker" not in c["naam"].lower()), competitions[0])
    current_guid = league_comp["guid"]
    current_season = current_guid[4:8]  # 'BVBL' + season + rest, e.g. 'BVBL26279130...' -> '2627'
    last_season = _previous_season(current_season)

    # strip season, then strip the trailing '<2-digit tier><1-letter pool>' (e.g. '31A')
    base = current_guid.replace(current_season, last_season, 1)[:-3]

    all_matches = []
    for tier in tiers:
        for letter in pool_letters:
            df = _get_competition_matches(f"{base}{tier}{letter}")
            if df.empty:
                continue
            ours = df[(df["HomeTeamGuid"] == team_guid) | (df["AwayTeamGuid"] == team_guid)]
            if not ours.empty:
                all_matches.append(ours)
        if all_matches:
            break  # a team belongs to exactly one pool/tier per season -- stop once found

    games_df = (
        pd.concat(all_matches, ignore_index=True) if all_matches
        else pd.DataFrame(columns=["Guid", "Date", "HomeTeam", "AwayTeam", "HomeTeamGuid", "AwayTeamGuid"])
    )

    if not games_df.empty:
        games_df["Opponent"] = games_df.apply(
            lambda r: r["AwayTeam"] if r["HomeTeamGuid"] == team_guid else r["HomeTeam"], axis=1)
        games_df["IsHome"] = games_df["HomeTeamGuid"] == team_guid
        games_df = games_df.sort_values("Date").reset_index(drop=True)

    return team_name, last_season, games_df

## 2. `get_game_overview` — open a single game's link and read its high-level details

In [23]:
def get_game_overview(page, game_guid: str) -> dict:
    """Open https://.../games/{game_guid} and return competition, date, both teams
    (name + club/team guids parsed from their profile links), and the final score."""
    page.goto(f"https://app.basketballstatsvlaanderen.be/games/{game_guid}",
              wait_until="networkidle", timeout=30000)
    page.wait_for_timeout(600)

    comp_link = page.locator("p:has(i.bi-trophy) a").first
    competition = comp_link.inner_text().strip() if comp_link.count() else None

    date_p = page.locator("p:has(i.bi-calendar-event)").first
    date_text = date_p.inner_text().strip() if date_p.count() else None

    team_links = page.locator(".row.text-center.mb-2 a")
    home_href = team_links.nth(0).get_attribute("href") or ""
    home_name = team_links.nth(0).inner_text().strip()
    away_href = team_links.nth(1).get_attribute("href") or ""
    away_name = team_links.nth(1).inner_text().strip()

    score_divs = page.locator(".row.text-center.mb-3 .col-6")
    home_score = score_divs.nth(0).inner_text().strip() if score_divs.count() > 0 else None
    away_score = score_divs.nth(1).inner_text().strip() if score_divs.count() > 1 else None

    def _split_club_team(href):
        # href looks like /clubs/{clubGuid}/{teamGuid}
        parts = href.strip("/").split("/")
        return (parts[1], parts[2]) if len(parts) >= 3 else (None, None)

    home_club_guid, home_team_guid = _split_club_team(home_href)
    away_club_guid, away_team_guid = _split_club_team(away_href)

    return {
        "GameGuid": game_guid,
        "Competition": competition,
        "DateText": date_text,
        "HomeTeam": home_name,
        "HomeClubGuid": home_club_guid,
        "HomeTeamGuid": home_team_guid,
        "AwayTeam": away_name,
        "AwayClubGuid": away_club_guid,
        "AwayTeamGuid": away_team_guid,
        "HomeScore": home_score,
        "AwayScore": away_score,
    }

## 3. `get_game_team_players` — open `/home` and `/away`, keep only our team's players

In [24]:
def get_game_team_players(page, game_guid: str, our_club_guid: str, overview: dict | None = None) -> pd.DataFrame:
    """Open the box-score sub-page (/home or /away, whichever matches our_club_guid) and
    return one row per player who appeared in the game, with their per-game stats."""
    if overview is None:
        overview = get_game_overview(page, game_guid)

    side = "home" if overview["HomeClubGuid"] == our_club_guid else "away"

    page.goto(f"https://app.basketballstatsvlaanderen.be/games/{game_guid}/{side}",
              wait_until="networkidle", timeout=30000)
    page.wait_for_timeout(600)

    tab = page.locator(".tabulator").first
    if tab.count() == 0:
        return pd.DataFrame()

    headers = tab.locator(".tabulator-col[tabulator-field]")
    fields = [headers.nth(i).get_attribute("tabulator-field") for i in range(headers.count())]
    fields = [f for f in fields if f != "Guid"]  # that column is just an icon, no data

    recs = []
    for row in tab.locator(".tabulator-row").all():
        rec = {f: (row.locator(f'[tabulator-field="{f}"]').inner_text().strip() or None) for f in fields}
        if rec.get("Name"):  # drop the team-totals row (blank name)
            recs.append(rec)

    df = pd.DataFrame(recs)
    if not df.empty:
        df.insert(0, "Side", side)
    return df

## 4. Put it together: one DataFrame — team / competition / game details / player details

In [25]:
def _get_club_name(club_guid: str) -> str:
    """Plain club name (e.g. 'BBC Haantjes Certifisc Oudenaarde'), without the team suffix
    that team_name carries (e.g. '... HSE B')."""
    r = requests.get(f"{BASE_URL}/api/clubs/{club_guid}", timeout=20)
    r.raise_for_status()
    return r.json()["club"]["Name"]


def build_last_season_player_stats(club_guid: str, team_guid: str, headless: bool = True) -> pd.DataFrame:
    """Run functions 1-3 for every game of a team's last season and stack the results."""

    club_name = _get_club_name(club_guid)
    team_name, season, games_df = get_last_season_games(club_guid, team_guid)
    print(f"Club: {club_name} | Team: {team_name} | Last season: {season} | Games found: {len(games_df)}")

    if games_df.empty:
        return pd.DataFrame()

    def _run():
        import asyncio
        # ipykernel forces WindowsSelectorEventLoopPolicy (needed for zmq), but Playwright's
        # driver subprocess needs WindowsProactorEventLoopPolicy -- swap it just in this
        # worker thread, for the duration of this call.
        original_policy = asyncio.get_event_loop_policy()
        if hasattr(asyncio, "WindowsProactorEventLoopPolicy"):
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

        all_rows = []
        try:
            with sync_playwright() as p:
                browser = p.chromium.launch(headless=headless)
                page = browser.new_page()

                for _, game in games_df.iterrows():
                    overview = get_game_overview(page, game["Guid"])
                    players = get_game_team_players(page, game["Guid"], club_guid, overview=overview)
                    if players.empty:
                        continue
                    players = players.copy()
                    players.insert(0, "Club", club_name)
                    players.insert(1, "Team", team_name)
                    players.insert(2, "Competition", overview["Competition"])
                    players.insert(3, "GameGuid", game["Guid"])
                    players.insert(4, "Date", overview["DateText"])
                    players.insert(5, "Opponent", game.get("Opponent"))
                    players.insert(6, "HomeScore", overview["HomeScore"])
                    players.insert(7, "AwayScore", overview["AwayScore"])
                    all_rows.append(players)

                browser.close()
        finally:
            asyncio.set_event_loop_policy(original_policy)

        print(f"Games with player stats: {len(all_rows)} / {len(games_df)}")
        return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(_run).result()

In [7]:
player_stats_df = build_last_season_player_stats(CLUB_GUID, TEAM_GUID, headless=HEADLESS)
player_stats_df

Club: BBC Haantjes Certifisc Oudenaarde | Team: BBC Haantjes Certifisc Oudenaarde HSE B | Last season: 2526 | Games found: 26


Games with player stats: 26 / 26


,Club,Team,Competition,GameGuid,Date,Opponent,HomeScore,AwayScore,Side,Number,Name,NormalizedMinutes,PlusMinus,Faults,TotalScore,ThreePointers,FieldGoals,FreeThrows,TotalScorePerMinute,PlusMinusPerMinute
0,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BCA,za 13 september om 22:15,BBC Hotshots Destelbergen HSE B,43,66,home,5,Rémi Bauters,13,-4,5,0,0p,0p,0p,0.00,-0.30
1,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BCA,za 13 september om 22:15,BBC Hotshots Destelbergen HSE B,43,66,home,6,Thomas Goossens,15,-7,0,0,0p,0p,0p,0.00,-0.50
2,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BCA,za 13 september om 22:15,BBC Hotshots Destelbergen HSE B,43,66,home,7,Mauro Boudringhien,7,-5,0,0,0p,0p,0p,0.00,-0.70
3,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BCA,za 13 september om 22:15,BBC Hotshots Destelbergen HSE B,43,66,home,8,Mathieu De Clercq,23,-18,3,12,12p,0p,0p,0.50,-0.80
4,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BCA,za 13 september om 22:15,BBC Hotshots Destelbergen HSE B,43,66,home,9,Matti Bogaert,27,-12,4,5,3p,0p,2p,0.20,-0.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BKC,zo 26 april om 17:00,Amon Jeugd Gentson HSE D,97,65,away,10,Emiel Vanderhelstraeten,27,-12,4,6,0p,2p,4p,0.20,-0.40
276,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BKC,zo 26 april om 17:00,Amon Jeugd Gentson HSE D,97,65,away,11,Thomas Viaene,27,-30,3,7,0p,6p,1p,0.30,-1.10
277,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BKC,zo 26 april om 17:00,Amon Jeugd Gentson HSE D,97,65,away,12,Brecht Van Glabeke,26,-22,2,11,3p,8p,0p,0.40,-0.80
278,BBC Haantjes Certifisc Oudenaarde,BBC Haantjes Certifisc Oudenaarde HSE B,3e Prov. Heren Oost-Vlaanderen B,BVBL25269130OVHSE31BKC,zo 26 april om 17:00,Amon Jeugd Gentson HSE D,97,65,away,13,Dylann Bufkens,20,0,3,0,0p,0p,0p,0.00,0.00


**On performance:** this runs one Playwright page-load per game (overview + home + away = 3 navigations), so for ~25-30 games expect roughly a minute or so. Set `HEADLESS = False` above if you want to watch it work.

**Scope:** `get_last_season_games` currently covers league games only (via the competition/pool page). Cup games -- like the `BVBL25269130BOVHSEVR03` example used to verify functions 2/3 -- live in a separate bracket-style competition structure that doesn't expose a flat match list the same way, so they aren't picked up automatically yet. If you want those included too, they can be added as a separate lookup once needed.

In [26]:
import re as _re


def _safe_filename(name: str) -> str:
    return _re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


if not player_stats_df.empty:
    club_name = player_stats_df["Club"].iloc[0]
    team_name = player_stats_df["Team"].iloc[0]
    filename = f"{_safe_filename(club_name)}__{_safe_filename(team_name)}_last_season_player_stats.csv"
    player_stats_df.to_csv(filename, index=False)
    print(f"Saved {len(player_stats_df)} rows to {filename}")
else:
    print("Nothing to export -- player_stats_df is empty.")

Nothing to export -- player_stats_df is empty.


## 5. Run for a different club/team

The main config cell above (`CLUB_GUID`/`TEAM_GUID`) stays pointed at HAANTJES. To pull the same
report for any other team without touching that, set `OTHER_CLUB_GUID`/`OTHER_TEAM_GUID` below
and run this cell -- e.g. any team from `.../clubs/{clubGuid}/{teamGuid}`. Defaults to the
Mibac Middelkerke example used to verify the tier/pool-promotion fix.

In [29]:
OTHER_CLUB_GUID = "BVBL1276"
OTHER_TEAM_GUID = "BVBL1276HSE%20%201"

other_player_stats_df = build_last_season_player_stats(OTHER_CLUB_GUID, OTHER_TEAM_GUID, headless=HEADLESS)
other_player_stats_df

Club: Basket Midwest BP Tielt | Team: Basket Midwest Izegem HSE A | Last season: 2526 | Games found: 0


""


In [28]:
# Optional: save this other team's stats to CSV alongside this notebook
if not other_player_stats_df.empty:
    other_club_name = other_player_stats_df["Club"].iloc[0]
    other_team_name = other_player_stats_df["Team"].iloc[0]
    other_filename = f"{_safe_filename(other_club_name)}__{_safe_filename(other_team_name)}_last_season_player_stats.csv"
    other_player_stats_df.to_csv(other_filename, index=False)
    print(f"Saved {len(other_player_stats_df)} rows to {other_filename}")
else:
    print("Nothing to export -- other_player_stats_df is empty.")

Saved 221 rows to Mibac_Middelkerke__Mibac_Middelkerke_HSE_B_last_season_player_stats.csv
